In [255]:
import os
import numpy as np
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from sympy.codegen.cfunctions import log10
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch 
from torchvision import transforms
from tqdm import tqdm
from PIL import Image
import time
from sklearn.metrics import accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torchvision.models as tv_models
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import torchvision.models as models
import cv2
from typing import Dict, List
import csv
from dataclasses import dataclass
import argparse
import json
from typing import Dict, List, Optional, Tuple

In [256]:
if torch.cuda.is_available():
    print(f"GPU kullanlabilir. Cihaz: {torch.cuda.get_device_name(0)}")
else:
    print("GPU kullanılabilir değil, CPU kullanılacak.")

GPU kullanlabilir. Cihaz: NVIDIA GeForce GTX 1650 Ti


In [257]:
CONFIG = {
    "root":      r"C:\Users\tuann\Desktop\Sanal Kıyafet Deneme Uygulaması\datasets",
    "train_csv": r"C:\Users\tuann\Desktop\Sanal Kıyafet Deneme Uygulaması\datasets\train_pairs_temiz.csv",
 
    # Stage 1: 128px pretrain
    #"epochs":       30,
    #"batch_size":   16,
    #"img_size":     128,
    #"lr":           2e-4,
 
    # Stage 2: 256px fine-tune (train() cagrisinda degistir)
     "epochs":     30,
     "batch_size": 4,
     "img_size":   256,
     "lr":         1e-4,
 
    "out_dir":      "models",
    "log_dir":      "logs",
    "json_dir":     "datasets/train/openpose_json",
    "graphics_dir": "graphics",
    "use_amp":      False,   # Mixed precision — VRAM tasarrufu
 
    # Stage 2 icin onceki modeli yukle
    "pretrain_g": "models/generator_4.pth",
    "pretrain_d": "models/discriminator_4.pth"
}

In [258]:
def load_openpose_keypoints(json_path: str, img_w: int, img_h: int, orig_w: int = 762, orig_h: int = 1016) -> Optional[np.ndarray]:
    """
    OpenPose JSON'dan keypoint okur ve img boyutuna olcekler.
    Donus: (25, 3) array [x, y, conf] veya None
    """
    try:
        with open(json_path, "r") as f:
            data = json.load(f)
        people = data.get("people", [])
        if not people:
            return None
        kps_flat = people[0].get("pose_keypoints_2d", [])
        if len(kps_flat) < 75:
            return None
        kps = np.array(kps_flat, dtype=np.float32).reshape(-1, 3)  # (25, 3)
        # Olcekle
        kps[:, 0] = kps[:, 0] / orig_w * img_w
        kps[:, 1] = kps[:, 1] / orig_h * img_h
        return kps
    except Exception:
        return None

In [259]:
def _clean_mask_train(mask: np.ndarray) -> np.ndarray:
    m = (mask > 0).astype(np.uint8) * 255
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN,  np.ones((3, 3), np.uint8), iterations=1)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8), iterations=2)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n > 1:
        keep = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        m = np.where(labels == keep, 255, 0).astype(np.uint8)
    return m
 
 
def warp_cloth_training(cloth_bgr: np.ndarray, cloth_mask: np.ndarray,
                         keypoints: np.ndarray, img_size: int,
                         widen_ratio: float = 0.25) -> Tuple[np.ndarray, np.ndarray]:
    """
    Egitim sirasinda cloth'u OpenPose keypoint'lere gore warp eder.
    OpenPose indeksleri: 2=sol omuz, 5=sag omuz, 8=sol kalca, 11=sag kalca
    """
    try:
        # OpenPose indeksleri
        ls = keypoints[2, :2].copy()   # sol omuz
        rs = keypoints[5, :2].copy()   # sag omuz
        lh = keypoints[8, :2].copy()   # sol kalca
        rh = keypoints[11, :2].copy()  # sag kalca
 
        # Dusuk guven skorlarini kontrol et
        if keypoints[2, 2] < 0.3 or keypoints[5, 2] < 0.3:
            return cloth_bgr, cloth_mask
        if keypoints[8, 2] < 0.2 or keypoints[11, 2] < 0.2:
            return cloth_bgr, cloth_mask
 
    except (IndexError, TypeError):
        return cloth_bgr, cloth_mask
 
    hp = wp = img_size
    src_mask = _clean_mask_train(cloth_mask)
    ys, xs   = np.where(src_mask > 0)
    if len(xs) == 0:
        return cloth_bgr, cloth_mask
 
    x, y, bw, bh = cv2.boundingRect(np.stack([xs, ys], axis=1))
    if bw < 4 or bh < 4:
        return cloth_bgr, cloth_mask
 
    # Sol/sag duzelt
    if ls[0] < rs[0]:
        ls, rs = rs.copy(), ls.copy()
    if lh[0] < rh[0]:
        lh, rh = rh.copy(), lh.copy()
 
    shoulder_vec   = rs - ls
    hip_vec        = rh - lh
    shoulder_width = np.linalg.norm(shoulder_vec)
    min_width      = wp * 0.38
 
    if shoulder_width < min_width:
        sf   = min_width / max(shoulder_width, 1)
        cx   = (ls[0] + rs[0]) / 2
        cy_s = (ls[1] + rs[1]) / 2
        cy_h = (lh[1] + rh[1]) / 2
        ls   = np.array([cx + (ls[0] - cx) * sf, cy_s], dtype=np.float32)
        rs   = np.array([cx + (rs[0] - cx) * sf, cy_s], dtype=np.float32)
        lh   = np.array([cx + (lh[0] - cx) * sf, cy_h], dtype=np.float32)
        rh   = np.array([cx + (rh[0] - cx) * sf, cy_h], dtype=np.float32)
        shoulder_vec = rs - ls
        hip_vec      = rh - lh
 
    lsw = ls - shoulder_vec * widen_ratio
    rsw = rs + shoulder_vec * widen_ratio
    lhw = lh - hip_vec * (widen_ratio * 0.4)
    rhw = rh + hip_vec * (widen_ratio * 0.4)
 
    src_pts = np.array([
        [x,      y     ],
        [x + bw, y     ],
        [x + bw, y + bh],
        [x,      y + bh],
    ], dtype=np.float32)
 
    dst_pts = np.array([lsw, rsw, rhw, lhw], dtype=np.float32)
 
    try:
        M           = cv2.getPerspectiveTransform(src_pts, dst_pts)
        warped_img  = cv2.warpPerspective(cloth_bgr,  M, (wp, hp), flags=cv2.INTER_LINEAR)
        warped_mask = cv2.warpPerspective(src_mask,   M, (wp, hp), flags=cv2.INTER_NEAREST)
        warped_mask = (warped_mask > 0).astype(np.uint8) * 255
    except cv2.error:
        return cloth_bgr, cloth_mask
 
    torso_poly  = np.array([lsw, rsw, rhw, lhw], dtype=np.int32)
    torso_roi   = np.zeros((hp, wp), dtype=np.uint8)
    cv2.fillConvexPoly(torso_roi, torso_poly, 255)
    torso_roi   = cv2.dilate(torso_roi, np.ones((20, 20), np.uint8), iterations=1)
    warped_mask = cv2.bitwise_and(warped_mask, torso_roi)
    warped_img  = cv2.bitwise_and(warped_img, warped_img, mask=warped_mask)
 
    # Maske bos veya cok kucukse orijinali dondur
    if warped_mask.sum() < 1000:
        return cloth_bgr, cloth_mask
 
    return warped_img, warped_mask
 

In [260]:
def read_rgb(path: str, size: int) -> np.ndarray:
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)
    return img
 
 
def read_mask(path: str, size: int) -> np.ndarray:
    mask = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(path)
    mask = cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)
    return (mask > 127).astype(np.uint8) * 255
 
 
def to_tensor(img_rgb: np.ndarray) -> torch.Tensor:
    t = img_rgb.astype(np.float32) / 127.5 - 1.0
    t = np.transpose(t, (2, 0, 1))
    return torch.from_numpy(t)
 
 
def mask_to_tensor(mask: np.ndarray) -> torch.Tensor:
    t = mask.astype(np.float32) / 255.0
    return torch.from_numpy(t).unsqueeze(0)

In [261]:
class PairTryOnDataset(Dataset):
    REQUIRED = {
        "person_image", "cloth_image", "cloth_mask",
        "pose_map", "agnostic_image", "target_image",
    }
 
    def __init__(self, csv_path: str, root: str, size: int, json_dir: str):
        self.root     = root
        self.size     = size
        self.json_dir = json_dir  # tam yol
        self.rows: List[Dict[str, str]] = []
 
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.rows.append(row)
 
        if not self.rows:
            raise ValueError("CSV bos!")
 
        missing = self.REQUIRED - set(self.rows[0].keys())
        if missing:
            raise ValueError(f"CSV eksik kolonlar: {sorted(missing)}")
 
    def __len__(self) -> int:
        return len(self.rows)
 
    def _abs(self, rel: str) -> str:
        return os.path.join(self.root, rel.replace("/", os.sep))
 
    def _json_path(self, person_image_rel: str) -> str:
        stem = os.path.splitext(os.path.basename(person_image_rel))[0]
        return os.path.join(self.json_dir, f"{stem}_keypoints.json")
 
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row  = self.rows[idx]
        size = self.size
 
        person_rgb   = read_rgb (self._abs(row["person_image"]),   size)
        cloth_rgb    = read_rgb (self._abs(row["cloth_image"]),    size)
        cloth_mask   = read_mask(self._abs(row["cloth_mask"]),     size)
        agnostic_rgb = read_rgb (self._abs(row["agnostic_image"]), size)
        target_rgb   = read_rgb (self._abs(row["target_image"]),   size)
 
        # OpenPose keypoint oku ve warp uygula
        json_path = self._json_path(row["person_image"])
        keypoints = load_openpose_keypoints(json_path, size, size)
 
        cloth_bgr = cv2.cvtColor(cloth_rgb, cv2.COLOR_RGB2BGR)
        if keypoints is not None:
            warped_bgr, warped_mask = warp_cloth_training(
                cloth_bgr, cloth_mask, keypoints, size
            )
        else:
            warped_bgr, warped_mask = cloth_bgr, cloth_mask
 
        warped_rgb = cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2RGB)
 
        return {
            "person":     to_tensor(person_rgb),
            "cloth":      to_tensor(warped_rgb),
            "cloth_mask": mask_to_tensor(warped_mask),
            "agnostic":   to_tensor(agnostic_rgb),
            "target":     to_tensor(target_rgb),
            # pose map yerine agnostic kullaniyoruz (pose map format farki nedeniyle)
            "pose":       to_tensor(agnostic_rgb),
        }
 

In [262]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)
 
 
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        feat = self.conv(x)
        return feat, self.pool(feat)
 
 
class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(out_ch * 2, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = nn.functional.interpolate(x, size=skip.shape[-2:],
                                           mode="bilinear", align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))
 

In [263]:
class TryOnGenerator(nn.Module):
    def __init__(self, in_ch=10, base=64):
        super().__init__()
        self.d1  = Down(in_ch,    base)
        self.d2  = Down(base,     base*2)
        self.d3  = Down(base*2,   base*4)
        self.mid = ConvBlock(base*4, base*8)
        self.u3  = Up(base*8, base*4)
        self.u2  = Up(base*4, base*2)
        self.u1  = Up(base*2, base)
        self.out = nn.Conv2d(base, 3, 1)
        self.tanh = nn.Tanh()
 
    def forward(self, agnostic, cloth, cloth_mask, pose):
        x = torch.cat([agnostic, cloth, cloth_mask, pose], dim=1)
        s1, x = self.d1(x)
        s2, x = self.d2(x)
        s3, x = self.d3(x)
        x = self.mid(x)
        x = self.u3(x, s3)
        x = self.u2(x, s2)
        x = self.u1(x, s1)
        return self.tanh(self.out(x))

In [264]:
class PatchDiscriminator(nn.Module):
    def __init__(self, in_ch=13, base=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,    base,   4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base,    base*2, 4, 2, 1),
            nn.BatchNorm2d(base*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base*2,  base*4, 4, 2, 1),
            nn.BatchNorm2d(base*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base*4,  1,      4, 1, 1),
        )
    def forward(self, agnostic, cloth, cloth_mask, pose, image):
        return self.net(torch.cat([agnostic, cloth, cloth_mask, pose, image], dim=1))
 
 
def sobel_edge_map(img):
    gray = 0.299*img[:,0:1] + 0.587*img[:,1:2] + 0.114*img[:,2:3]
    kx = torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]],
                       dtype=img.dtype, device=img.device).view(1,1,3,3)
    ky = torch.tensor([[1,2,1],[0,0,0],[-1,-2,-1]],
                       dtype=img.dtype, device=img.device).view(1,1,3,3)
    ex = nn.functional.conv2d(gray, kx, padding=1)
    ey = nn.functional.conv2d(gray, ky, padding=1)
    return torch.sqrt(ex*ex + ey*ey + 1e-6)

In [265]:
graphics_pth = r"graphics"
if not os.path.exists(graphics_pth):
    os.makedirs(graphics_pth)
graphic_files = [f for f in os.listdir(graphics_pth) if f.startswith("graph") and f.endswith("png")]
if not graphic_files:
    graphic_filename = "graph_1.png"
else:
    graphic_max_suffix = max([int(f.split('_')[-1].split('.')[0]) for f in graphic_files])
    graphic_filename = f"graph_{graphic_max_suffix + 1}.png"
graphic_save_path = os.path.join(graphics_pth, graphic_filename)

log_pth = r"logs"
log_files = [f for f in os.listdir(log_pth) if f.startswith("log") and f.endswith("csv")]
if not log_files:
    log_filename = "log_1.csv"
else:
    log_max_suffix = max([int(f.split('_')[-1].split('.')[0]) for f in log_files])
    log_filename = f"log_{log_max_suffix + 1}.csv"
log_save_path = os.path.join(log_pth, log_filename)

In [266]:
@dataclass
class LossWeights:
    adv:  float = 0.3
    l1:   float = 15.0
    edge: float = 2.0
    perc: float = 2.0   # VGG perceptual
    tv:   float = 0.05  # Total variation
 
 
def build_vgg_extractor(device):
    """VGG19 ilk 16 katman — perceptual loss icin."""
    vgg = tv_models.vgg19(weights=tv_models.VGG19_Weights.IMAGENET1K_V1)
    extractor = vgg.features[:16].eval().to(device)
    for p in extractor.parameters():
        p.requires_grad = False
    return extractor
 
 
def perceptual_loss(fake, real, vgg_extractor):
    """VGG feature uzayinda L1 farki."""
    # [-1,1] -> [0,1] -> ImageNet normalize
    mean = torch.tensor([0.485, 0.456, 0.406], device=fake.device).view(1,3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=fake.device).view(1,3,1,1)
    f = ((fake  + 1) / 2 - mean) / std
    r = ((real  + 1) / 2 - mean) / std
    return F.l1_loss(vgg_extractor(f), vgg_extractor(r))
 
 
def tv_loss(x):
    """Total variation — lekelenme/noise azaltir."""
    return (
        (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean() +
        (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
    )
 
 
def get_next_filename(folder, prefix, ext):
    os.makedirs(folder, exist_ok=True)
    files = [f for f in os.listdir(folder)
             if f.startswith(prefix+"_") and f.endswith("."+ext)]
    if not files:
        return os.path.join(folder, f"{prefix}_1.{ext}")
    max_n = max(int(f.replace(prefix+"_","").replace("."+ext,"")) for f in files)
    return os.path.join(folder, f"{prefix}_{max_n+1}.{ext}")
 
 
def save_sample(batch, fake, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    target  = batch["target"][0].detach().cpu().numpy()
    fake_np = fake[0].detach().cpu().numpy()
    target  = ((np.transpose(target,  (1,2,0))+1.0)*127.5).clip(0,255).astype(np.uint8)
    fake_np = ((np.transpose(fake_np, (1,2,0))+1.0)*127.5).clip(0,255).astype(np.uint8)
    vis = np.concatenate([target, fake_np], axis=1)
    cv2.imwrite(out_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
 
 
def save_loss_graph(log_path, graph_path):
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
    axes[0].plot(df["epoch"], df["g_total"], label="G Total")
    axes[0].plot(df["epoch"], df["g_adv"],   label="G Adv")
    axes[0].plot(df["epoch"], df["g_l1"],    label="G L1")
    axes[0].plot(df["epoch"], df["g_edge"],  label="G Edge")
    axes[0].set_title("Generator Loss"); axes[0].legend(); axes[0].grid(True)
 
    axes[1].plot(df["epoch"], df["g_perc"], label="Perceptual", color="purple")
    axes[1].plot(df["epoch"], df["g_tv"],   label="TV",         color="orange")
    axes[1].set_title("Perceptual + TV Loss"); axes[1].legend(); axes[1].grid(True)
 
    axes[2].plot(df["epoch"], df["d_total"], label="D Total", color="red")
    axes[2].set_title("Discriminator Loss"); axes[2].legend(); axes[2].grid(True)
 
    plt.tight_layout()
    plt.savefig(graph_path, dpi=150)
    plt.close()
    print(f"Grafik kaydedildi: {graph_path}")

In [267]:
def train_one_epoch(loader, g, d, g_opt, d_opt, bce, l1_fn, weights,
                    device, vgg_extractor, scaler=None):
    g.train(); d.train()
    meter = {"g_total":0., "g_adv":0., "g_l1":0., "g_edge":0.,
             "g_perc":0., "g_tv":0., "d_total":0.}
    n = 0
    use_amp = scaler is not None
 
    for batch in loader:
        agnostic   = batch["agnostic"].to(device)
        cloth      = batch["cloth"].to(device)
        cloth_mask = batch["cloth_mask"].to(device)
        pose       = batch["pose"].to(device)
        target     = batch["target"].to(device)
 
        # ── Discriminator (her 2 batch'te bir) ──
        if n % 2 == 0:
            d_opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                with torch.no_grad():
                    fake_d = g(agnostic, cloth, cloth_mask, pose)
                pred_real = d(agnostic, cloth, cloth_mask, pose, target)
                pred_fake = d(agnostic, cloth, cloth_mask, pose, fake_d)
                d_loss = 0.5 * (
                    bce(pred_real, torch.ones_like(pred_real)) +
                    bce(pred_fake, torch.zeros_like(pred_fake))
                )
            if use_amp:
                scaler.scale(d_loss).backward()
                scaler.step(d_opt)
                scaler.update()
            else:
                d_loss.backward()
                d_opt.step()
            meter["d_total"] += d_loss.item()
 
        # ── Generator ──
        g_opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=use_amp):
            fake        = g(agnostic, cloth, cloth_mask, pose)
            pred_fake_g = d(agnostic, cloth, cloth_mask, pose, fake)
            g_adv  = bce(pred_fake_g, torch.ones_like(pred_fake_g))
            g_l1   = l1_fn(fake, target)
            g_edge = l1_fn(sobel_edge_map(fake), sobel_edge_map(target))
            g_perc = perceptual_loss(fake, target, vgg_extractor)
            g_tv   = tv_loss(fake)
            g_total = (weights.adv  * g_adv  +
                       weights.l1   * g_l1   +
                       weights.edge * g_edge +
                       weights.perc * g_perc +
                       weights.tv   * g_tv)
        if use_amp:
            scaler.scale(g_total).backward()
            scaler.step(g_opt)
            scaler.update()
        else:
            g_total.backward()
            g_opt.step()
 
        meter["g_total"] += g_total.item()
        meter["g_adv"]   += g_adv.item()
        meter["g_l1"]    += g_l1.item()
        meter["g_edge"]  += g_edge.item()
        meter["g_perc"]  += g_perc.item()
        meter["g_tv"]    += g_tv.item()
        n += 1
 
    return {k: v/max(n,1) for k,v in meter.items()}
 
 
 
def train(config=CONFIG):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Cihaz: {device}")
 
    model_dir    = config["out_dir"]
    graphics_dir = config["graphics_dir"]
    log_dir      = config["log_dir"]
    json_dir     = os.path.join(config["root"], config["json_dir"])
    use_amp      = config.get("use_amp", False) and device.type == "cuda"
 
    for d in [model_dir, graphics_dir, log_dir]:
        os.makedirs(d, exist_ok=True)
 
    log_path   = get_next_filename(log_dir,      "log",           "csv")
    graph_path = get_next_filename(graphics_dir, "graph",         "png")
    g_path     = get_next_filename(model_dir,    "generator",     "pth")
    d_path     = get_next_filename(model_dir,    "discriminator", "pth")
 
    print(f"Log    : {log_path}")
    print(f"Grafik : {graph_path}")
    print(f"G model: {g_path}")
    print(f"D model: {d_path}")
    print(f"AMP    : {use_amp}")
 
    dataset = PairTryOnDataset(
        config["train_csv"], config["root"],
        config["img_size"],  json_dir
    )
    loader = DataLoader(dataset, batch_size=config["batch_size"],
                        shuffle=True, num_workers=0, pin_memory=False)
    print(f"Dataset: {len(dataset)} ornek, {len(loader)} batch")
 
    # VGG extractor
    vgg_extractor = build_vgg_extractor(device)
    print("VGG extractor hazir.")
 
    g     = TryOnGenerator().to(device)
    d_net = PatchDiscriminator().to(device)
 
    # Onceki checkpoint yukle (Stage 2 icin)
    if config.get("pretrain_g") and os.path.isfile(config["pretrain_g"]):
        g.load_state_dict(torch.load(config["pretrain_g"], map_location=device))
        print(f"Generator yuklendi: {config['pretrain_g']}")
    if config.get("pretrain_d") and os.path.isfile(config["pretrain_d"]):
        d_net.load_state_dict(torch.load(config["pretrain_d"], map_location=device))
        print(f"Discriminator yuklendi: {config['pretrain_d']}")
 
    g_opt = optim.Adam(g.parameters(),     lr=config["lr"],       betas=(0.5, 0.999))
    d_opt = optim.Adam(d_net.parameters(), lr=config["lr"]*0.25,  betas=(0.5, 0.999))
    bce   = nn.BCEWithLogitsLoss()
    l1_fn = nn.L1Loss()
    w     = LossWeights()
 
    # AMP scaler
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
 
    with open(log_path, "w", encoding="utf-8") as f:
        f.write("epoch,g_total,g_adv,g_l1,g_edge,g_perc,g_tv,d_total\n")
 
    best_l1    = float("inf")
    best_epoch = -1
 
    for epoch in range(1, config["epochs"]+1):
        m = train_one_epoch(loader, g, d_net, g_opt, d_opt, bce, l1_fn,
                            w, device, vgg_extractor, scaler)
        print(
            f"[{epoch:>3}/{config['epochs']}] "
            f"G={m['g_total']:.4f}  "
            f"(adv={m['g_adv']:.3f}, l1={m['g_l1']:.4f}, "
            f"edge={m['g_edge']:.3f}, perc={m['g_perc']:.3f}, tv={m['g_tv']:.4f})  "
            f"D={m['d_total']:.4f}"
        )
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"{epoch},{m['g_total']:.6f},{m['g_adv']:.6f},"
                    f"{m['g_l1']:.6f},{m['g_edge']:.6f},"
                    f"{m['g_perc']:.6f},{m['g_tv']:.6f},{m['d_total']:.6f}\n")
 
        if m["g_l1"] < best_l1:
            best_l1    = m["g_l1"]
            best_epoch = epoch
            torch.save(g.state_dict(),     g_path)
            torch.save(d_net.state_dict(), d_path)
            print(f"  --> En iyi model kaydedildi (epoch {epoch}, l1={best_l1:.4f})")
 
        if epoch % 5 == 0:
            sample_batch = next(iter(loader))
            with torch.no_grad():
                fake = g(
                    sample_batch["agnostic"].to(device),
                    sample_batch["cloth"].to(device),
                    sample_batch["cloth_mask"].to(device),
                    sample_batch["pose"].to(device),
                )
            save_sample(sample_batch, fake,
                        os.path.join(graphics_dir, f"sample_epoch_{epoch}.jpg"))
 
    print(f"\nEn iyi epoch: {best_epoch}, L1: {best_l1:.4f}")
    save_loss_graph(log_path, graph_path)
    print("Egitim tamamlandi.")
    print(f"\n--- STAGE 2 icin ---")
    print(f"CONFIG['pretrain_g'] = '{g_path}'")
    print(f"CONFIG['pretrain_d'] = '{d_path}'")
    print(f"CONFIG['img_size']   = 256")
    print(f"CONFIG['batch_size'] = 8")
    print(f"CONFIG['lr']         = 1e-4")
    print(f"CONFIG['epochs']     = 30")
 
 
train(CONFIG)
 

Cihaz: cuda
Log    : logs\log_37.csv
Grafik : graphics\graph_24.png
G model: models\generator_5.pth
D model: models\discriminator_5.pth
AMP    : False
Dataset: 11647 ornek, 2912 batch


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
